<a href="https://colab.research.google.com/github/fathinahnj/unet-plankton-segmentation/blob/main/kfold_as_default.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

ini coba2 saja 5-fold cross validation in an entirely new file. kita coba pake model baseline dulu. kalo ini berhasil, kita coba juga model yg augmented. kalo misal oke, jadikan ini sebagai default instead of the old files.

kalo nda berhasil, kita kembali ke old files, kita selipkan disitu kodenya

# Imports

In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="Vjof9rBzJ2QeB82ApBRi")
project = rf.workspace("planktonclassification").project("instance-t20")
version = project.version(4)
dataset = version.download("coco-segmentation")

loading Roboflow workspace...
loading Roboflow project...


In [ ]:
%%capture
import os
import cv2
import numpy as np
import tensorflow as tf
import pandas as pd

from pycocotools.coco import COCO
from sklearn.model_selection import KFold

from tensorflow.keras.optimizers import Adam
from tensorflow.keras import layers, Model

from tensorflow.keras.applications import (
    ResNet50,
    EfficientNetB4,
    MobileNetV2
)

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


# Path & Config

In [ ]:
BASE_PATH = "/content/drive/MyDrive/PLANKTON/data/instance/test"
DATA_PATH = "/content/instance-t20-4/valid"

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 8
EPOCHS = 20
KFOLD = 5
LR = 1e-5

MODEL_SAVE_DIR = os.path.join(BASE_PATH, "kfold_models")

os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

In [ ]:
ENCODERS = {
    "resnet50": ResNet50,
    "efficientnetb4": EfficientNetB4,
    "mobilenetv2": MobileNetV2
}

# Load COCO Dataset

In [ ]:
coco_json_path = os.path.join(DATA_PATH, "_annotations.coco.json")

coco = COCO(coco_json_path)

all_images = coco.dataset["images"]
all_annotations = coco.dataset["annotations"]
categories = coco.dataset["categories"]

image_ids = np.array([img["id"] for img in all_images])

loading annotations into memory...
Done (t=0.00s)
creating index...
index created!


In [ ]:
# shared coco object

coco_kfold_obj = COCO()
coco_kfold_obj.dataset = {
    "images":      all_images,
    "annotations": all_annotations,
    "categories":  categories
}
coco_kfold_obj.createIndex()

creating index...
index created!


# Image Path

In [ ]:
def get_image_path(file_name):
    return os.path.join(DATA_PATH, file_name)

# Mask Creation

In [ ]:
def create_mask(image_info, coco_instance):

    height = image_info["height"]
    width = image_info["width"]

    mask = np.zeros((height, width), dtype=np.uint8)

    ann_ids = coco_instance.getAnnIds(imgIds=image_info["id"])
    anns = coco_instance.loadAnns(ann_ids)

    for ann in anns:
        m = coco_instance.annToMask(ann)
        mask[m == 1] = 1

    return mask

In [ ]:
def load_image_and_mask(img_info, augment=False):
    """Load a single image + binary mask, optionally apply color augmentation."""
    img_path = os.path.join(DATA_PATH, img_info["file_name"])

    image = cv2.imread(img_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = cv2.resize(image, (IMG_SIZE, IMG_SIZE)) / 255.0

    mask = create_mask(img_info, coco_kfold_obj)
    mask = cv2.resize(mask, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)
    mask = np.expand_dims(mask, axis=-1)

    if augment:
        image = tf.image.random_brightness(image, max_delta=0.2)
        image = tf.image.random_contrast(image, lower=0.8, upper=1.2)
        image = tf.image.random_saturation(image, lower=0.8, upper=1.2)
        image = tf.clip_by_value(image, 0.0, 1.0).numpy()

    return image, mask

In [ ]:
def load_image_and_mask_augmented(img_info, apply_augmentation=True):

    img_path = os.path.join(data_path, img_info["file_name"])

    image = cv2.imread(img_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = cv2.resize(image, (IMG_SIZE, IMG_SIZE)) / 255.0

    mask = create_mask(img_info, coco_kfold_obj)
    mask = cv2.resize(mask, (IMG_SIZE, IMG_SIZE),
                      interpolation=cv2.INTER_NEAREST)

    mask = np.expand_dims(mask, axis=-1)

    if apply_augmentation:
        image = augment_color(image)

    return image, mask

# tf.data Pipeline

In [ ]:
def build_dataset(img_subset, augment=False, batch_size=BATCH_SIZE):
    """
    Build a tf.data.Dataset from a list of COCO image dicts.
    Loads lazily — no full numpy array materialised in memory.
    """
    def generator():
        for img_info in img_subset:
            image, mask = load_image_and_mask(img_info, augment=augment)
            yield image.astype(np.float32), mask.astype(np.float32)

    dataset = tf.data.Dataset.from_generator(
        generator,
        output_signature=(
            tf.TensorSpec(shape=(IMG_SIZE, IMG_SIZE, 3), dtype=tf.float32),
            tf.TensorSpec(shape=(IMG_SIZE, IMG_SIZE, 1), dtype=tf.float32)
        )
    )

    dataset = (
        dataset
        .shuffle(buffer_size=len(img_subset), reshuffle_each_iteration=True)
        .batch(batch_size)
        .prefetch(tf.data.AUTOTUNE)
    )

    return dataset

# Dice Score

In [ ]:
def dice_score(y_true, y_pred):

    intersection = np.sum(y_true * y_pred)
    union = np.sum(y_true) + np.sum(y_pred)

    return (2. * intersection + 1e-7) / (union + 1e-7)

# Loss Function

In [ ]:
def bce_dice_loss(y_true, y_pred):

    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)

    intersection = tf.reduce_sum(y_true * y_pred)
    union = tf.reduce_sum(y_true) + tf.reduce_sum(y_pred)

    dice = (2. * intersection + 1e-7) / (union + 1e-7)

    return bce + (1 - dice)

# UNet Builder + Encoders

In [ ]:
# Per-encoder: skip layer names + bridge layer name
ENCODER_CONFIG = {
    "resnet50": {
        "skips":  [
            "conv1_relu",
            "conv2_block3_out",
            "conv3_block4_out",
            "conv4_block6_out"
        ],
        "bridge": "conv5_block3_out",
        # ResNet bottleneck leaves output at 7x7 — needs an extra upsample to reach 224
        "extra_upsample": True
    },
    "efficientnetb4": {
        "skips":  [
            "block2a_activation",
            "block3a_activation",
            "block4a_activation"
            # block6a_activation intentionally excluded — spatial mismatch at 224x224
        ],
        "bridge": "top_activation",
        # After 3 skip upsamples we're at 56x56 — need 2 more to reach 224
        "extra_upsample": True
    },
    "mobilenetv2": {
        "skips":  [
            "block_1_expand_relu",
            "block_3_expand_relu",
            "block_6_expand_relu",
            "block_13_expand_relu"
        ],
        "bridge": "block_16_project",
        "extra_upsample": False
    }
}

In [ ]:
def build_unet_with_encoder(encoder_name):
    """
    Build a U-Net with the specified pretrained encoder.
    Supported: 'resnet50', 'efficientnetb4', 'mobilenetv2'.
    """
    config       = ENCODER_CONFIG[encoder_name]
    EncoderClass = ENCODERS[encoder_name]

    encoder = EncoderClass(
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_top=False,
        weights="imagenet"
    )

    skips  = [encoder.get_layer(name).output for name in config["skips"]]
    x      = encoder.get_layer(config["bridge"]).output

    # Decoder: upsample + skip connections
    for skip in reversed(skips):
        x = layers.UpSampling2D(size=(2, 2))(x)
        x = layers.Concatenate()([x, skip])
        x = layers.Conv2D(256, 3, padding="same", activation="relu")(x)
        x = layers.Conv2D(256, 3, padding="same", activation="relu")(x)

    # Extra upsampling blocks to bring spatial dims back to IMG_SIZE
    if config["extra_upsample"]:
        for filters in [128, 32]:
            x = layers.UpSampling2D(size=(2, 2))(x)
            x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
            x = layers.Conv2D(filters // 2, 3, padding="same", activation="relu")(x)

    outputs = layers.Conv2D(1, 1, activation="sigmoid")(x)

    return Model(encoder.input, outputs)

# K-Fold Setup

In [ ]:
kf = KFold(n_splits=KFOLD, shuffle=True, random_state=42)

encoder_results = {}

for encoder_name in ENCODERS.keys():

    print(f"\n{'='*40}")
    print(f"  ENCODER: {encoder_name.upper()}")
    print(f"{'='*40}")

    fold_scores = []

    for fold, (train_idx, val_idx) in enumerate(kf.split(image_ids)):

        print(f"\n----- Fold {fold + 1} / {KFOLD} -----")

        # Use sets for O(1) membership checks
        train_id_set = set(image_ids[train_idx])
        val_id_set   = set(image_ids[val_idx])

        train_imgs = [img for img in all_images if img["id"] in train_id_set]
        val_imgs   = [img for img in all_images if img["id"] in val_id_set]

        train_ds = build_dataset(train_imgs, augment=False, batch_size=BATCH_SIZE)
        val_ds   = build_dataset(val_imgs,   augment=False, batch_size=BATCH_SIZE)

        model = build_unet_with_encoder(encoder_name)
        model.compile(optimizer=Adam(LR), loss=bce_dice_loss)

        best_model_path = os.path.join(
            MODEL_SAVE_DIR,
            f"{encoder_name}_fold_{fold + 1}_best.h5"
        )

        callbacks = [
            EarlyStopping(
                monitor="val_loss",
                patience=5,
                restore_best_weights=True,
                verbose=1
            ),
            ModelCheckpoint(
                filepath=best_model_path,
                monitor="val_loss",
                save_best_only=True,
                verbose=1
            )
        ]

        model.fit(
            train_ds,
            validation_data=val_ds,
            epochs=EPOCHS,
            callbacks=callbacks,
            verbose=1
        )

        # Batched prediction over val set
        fold_dice = []

        for batch_imgs_np, batch_masks_np in val_ds:
            preds = model.predict(batch_imgs_np, verbose=0)
            preds_bin = (preds > 0.3).astype(np.uint8)

            for i in range(len(batch_masks_np)):
                d = dice_score(
                    batch_masks_np[i, :, :, 0].numpy(),
                    preds_bin[i, :, :, 0]
                )
                fold_dice.append(d)

        mean_dice = np.mean(fold_dice)
        fold_scores.append(mean_dice)
        print(f"Fold {fold + 1} Dice: {mean_dice:.4f}")

    encoder_results[encoder_name] = fold_scores
    print(f"\n{encoder_name} — Avg Dice: {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}")

# Result Comparison

In [ ]:
comparison = []

for encoder, scores in encoder_results.items():

    comparison.append({
        "Encoder": encoder,
        "Fold1": scores[0],
        "Fold2": scores[1],
        "Fold3": scores[2],
        "Fold4": scores[3],
        "Fold5": scores[4],
        "Average Dice": np.mean(scores),
        "Std Dice": np.std(scores)
    })

df = pd.DataFrame(comparison)
df = df.sort_values("Average Dice", ascending=False)

df